# Form Analyzer Data

Used to rename and organize custom dataset of videos

In [11]:
import os

def rename_videos(input_dir, output_dir, label, offset=0):
    """
    Renames and moves videos sequentially based on a label.
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Grab all .MOV or .mov files from the input directory
    files = [f for f in os.listdir(input_dir) if f.lower().endswith('.mov')]

    if not files:
        print(f"No .MOV files found in {input_dir}")
        return

    print(f"Found {len(files)} videos. Starting renaming for label: '{label}'...")

    for index, filename in enumerate(sorted(files)):
        modified_index = index + offset
        # Format the new filename with a padded sequential number (e.g., heel_strike_001.mp4)
        new_filename = f"{label}_{modified_index + 1:03d}.mov"
        
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, new_filename)
        os.rename(input_path, output_path)

    print(f"Success! All {label} videos saved to {output_dir}")

In [13]:
unlabeled_path = "datasets/custom_running_form/unlabeled"
labeled_path = "datasets/custom_running_form/"

rename_videos(unlabeled_path, labeled_path, "arms_loose")

Found 26 videos. Starting renaming for label: 'arms_loose'...
Success! All arms_loose videos saved to datasets/custom_running_form/


Next steps:

data modification (mirror, jitter)
split video data into train, test, val sets 
get keypoint time series dataset
create MLP for form analysis
train MLP and evaluate on validation set
modify MLP architecture as needed and repeat training

In [2]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS device found.")
else:
    print("MPS device not found.")

MPS device found.


In [3]:
from ultralytics import YOLO

model_path = "yolo11s-pose.pt"
model = YOLO(model_path).to(device)

In [7]:
import os

def process_results(results, filename):
    video_tensor_list = []
    for r in results:
        if len(r.keypoints) == 0 or len(r.boxes) == 0:
            continue

        # Grab the keypoints and bounding box for the FIRST detected person
        # keypoints.data shape is (1, 17, 3) -> [person_index, joint_index, (x, y, conf)]
        kpts = r.keypoints.data[0] 
        bbox_height = r.boxes.xywh[0][3] # [x_center, y_center, width, height]
        
        # Left Hip (11) and Right Hip (12)
        l_hip_x, l_hip_y = kpts[11][0], kpts[11][1]
        r_hip_x, r_hip_y = kpts[12][0], kpts[12][1]
        
        hip_center_x = (l_hip_x + r_hip_x) / 2.0
        hip_center_y = (l_hip_y + r_hip_y) / 2.0
        
        # Initialize an empty tensor for this frame's 17 normalized joints
        normalized_kpts = torch.zeros((17, 3))
        
        # Apply the localized mathematical normalization
        for i in range(17):
            x, y, conf = kpts[i]
            if conf > 0: # Only normalize if the joint is visible
                normalized_kpts[i][0] = (x - hip_center_x) / bbox_height
                normalized_kpts[i][1] = (y - hip_center_y) / bbox_height
            normalized_kpts[i][2] = conf # Keep visibility/confidence score as-is
            
        # Flatten the (17, 3) matrix into a 1D vector of length 51
        video_tensor_list.append(normalized_kpts.flatten())

    # Stack the list of 1D vectors into a final 2D PyTorch Tensor (T, 51)
    if len(video_tensor_list) > 0:
        final_timeseries_tensor = torch.stack(video_tensor_list)
        
        tensor_save_path = os.path.join("datasets", "custom_running_form", "keypoints", f"{filename.rsplit('.', 1)[0]}_features.pt")
        torch.save(final_timeseries_tensor, tensor_save_path)
        print(f"Successfully saved normalized feature tensor of shape {final_timeseries_tensor.shape}")

In [8]:

def get_video_keypoints(model, input_dir, output_dir):
    """
    Use pose estimator model to get keypoints for all .mov videos in input_dir
    Store keypoint files in output_dir
    """
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Grab all .MOV or .mov files from the input directory
    files = [f for f in os.listdir(input_dir) if f.lower().endswith('.mov')]

    if not files:
        print(f"No .MOV files found in {input_dir}")
        return

    print(f"Found {len(files)} videos. Performing pose estimation...")

    for filename in sorted(files):
        filepath = os.path.join(input_dir, filename)

        results = model(source=filepath, save=True, stream=True, conf=0.5, project="user_submissions", exist_ok=True)

        print("Pose results complete! Processing results...")
        
        process_results(results, filename)

        


In [9]:
get_video_keypoints(model, "datasets/custom_running_form/videos", "datasets/custom_running_form/keypoints")

Found 158 videos. Performing pose estimation...
Pose results complete! Processing results...

WARNING ⚠️ NMS time limit 2.050s exceeded
video 1/1 (frame 1/75) /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/custom_running_form/videos/arms_loose_001.mov: 384x640 1 person, 513.6ms
video 1/1 (frame 2/75) /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/custom_running_form/videos/arms_loose_001.mov: 384x640 1 person, 30.1ms
video 1/1 (frame 3/75) /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/custom_running_form/videos/arms_loose_001.mov: 384x640 1 person, 27.5ms
video 1/1 (frame 4/75) /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/custom_running_form/videos/arms_loose_001.mov: 384x640 1 person, 24.1ms
video 1/1 (frame 5/75) /Users/harrisonyork/Documents/AppliedML/FinalProject/datasets/custom_running_form/videos/arms_loose_001.mov: 384x640 1 person, 28.0ms
video 1/1 (frame 6/75) /Users/harrisonyork/Documents/AppliedML/FinalProject/da